# Render Setup Test

Verifies that the viewer state is correctly configured for rendering:
pipeline data, binary protocol, and frame updates.

Actual image/video rendering happens in the browser (Three.js);
this notebook tests the Python-side prerequisites.

In [ ]:
import json
import pathlib

import megane
from megane.pipeline import (
    AddBonds,
    AddLabels,
    LoadStructure,
    LoadTrajectory,
    Pipeline,
)

FIXTURES = (pathlib.Path(".").resolve().parent / "fixtures")
assert FIXTURES.exists()
print(f"megane v{megane.__version__}")

## Viewer state ready for snapshot rendering

In [ ]:
pipe = Pipeline()
s = pipe.add_node(LoadStructure(str(FIXTURES / "1crn.pdb")))
bonds = pipe.add_node(AddBonds(source="distance"))
labels = pipe.add_node(AddLabels(source="element"))
pipe.add_edge(s, bonds)
pipe.add_edge(s, labels)

viewer = megane.MolecularViewer()
viewer.set_pipeline(pipe)

# Verify pipeline is active
assert viewer._pipeline_enabled is True

# Verify pipeline JSON is valid
config = json.loads(viewer._pipeline_json)
assert config["version"] == 3
print(f"Pipeline config: {len(config['nodes'])} nodes, {len(config['edges'])} edges")

# Verify node snapshot data is populated
assert len(viewer._node_snapshots_data) > 0
print(f"Node snapshots: {len(viewer._node_snapshots_data)} entries")

viewer

## Frame data updates for animation rendering

In [ ]:
pipe_traj = Pipeline()
st = pipe_traj.add_node(LoadStructure(str(FIXTURES / "caffeine_water.pdb")))
traj = pipe_traj.add_node(LoadTrajectory(xtc=str(FIXTURES / "caffeine_water_vibration.xtc")))
pipe_traj.add_edge(st, traj)

v_traj = megane.MolecularViewer()
v_traj.set_pipeline(pipe_traj)

assert v_traj.total_frames > 1, f"Need multiple frames, got {v_traj.total_frames}"
print(f"Total frames: {v_traj.total_frames}")

# Iterate frames and verify frame data changes
frame_data_samples = {}
for i in [0, 5, 10]:
    v_traj.frame_index = i
    frame_data_samples[i] = v_traj._frame_data

assert frame_data_samples[0] != frame_data_samples[5], "Frame data did not change between frame 0 and 5"
assert frame_data_samples[5] != frame_data_samples[10], "Frame data did not change between frame 5 and 10"
print("Frame data updates correctly across frames")
v_traj

## Binary protocol verification

In [ ]:
# Verify MEGN magic bytes in node snapshot data
for node_id, data in viewer._node_snapshots_data.items():
    assert data[:4] == b"MEGN", f"Node {node_id}: expected MEGN magic, got {data[:4]}"
    print(f"Node {node_id}: MEGN header OK, {len(data)} bytes")

# Verify frame data also has MEGN header
assert v_traj._frame_data[:4] == b"MEGN", f"Frame data: expected MEGN magic, got {v_traj._frame_data[:4]}"
print(f"Frame data: MEGN header OK, {len(v_traj._frame_data)} bytes")

print("\nAll binary protocol checks passed")